# Smart Microscopy

**Cell 1** — Read focus positions from Overview job, enforce z-galvo, clear scan fields.
**Cell 2** — Read and visualise the scanning field.
**Cell 3** — Autofocus at the focus positions, fit Z plane.
**Cell 4** — Acquire at all tile positions with interpolated Z.

In [1]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path("controller/vendor/leica").resolve()))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import lasx
from lasx.scanning_templates import (
    TEMPLATE_BASE, TEMPLATE_XML, STRIPPED_XML,
)

# ── Connect to LAS X ────────────────────────────────────────────────
from LasxApi import PYLICamApiConnector as lasxApi

client = lasxApi.LasxApiClientPyModel
client.Connect("PythonClient")
client.PyApiClient.DelayInMilliseconds = 300
mode = client.PyApiSetApiInterfaceToUse.Model.ApiInterfaceToUse
client.PyApiSetApiInterfaceToUse.Model.ApiInterfaceToUse = (
    type(mode).Only_the_CAM_interface_is_used
)
assert lasx.ping(client), "LAS X not responding"

templates_dir = lasx.find_scanning_templates_dir()
print(f"Connected — templates: {templates_dir}")

Connected — templates: C:\Users\t.de\AppData\Roaming\Leica Microsystems\LAS X\MatrixScreener6\User_0\ScanningTemplates


## Cell 1 — Focus positions

Add **4 overview positions** in Navigator Expert (these define the focus reference plane), then run this cell.

In [3]:
LIMIT_MARGIN_UM = 500    # XY margin around the positions
Z_GALVO_MIN    = -200    # z-galvo limits (hardware-specific)
Z_GALVO_MAX    =  200
Z_WIDE_MIN     = -5000
Z_WIDE_MAX     =  5000

# ── Save and parse current template ──────────────────────────────────
lasx.save_experiment(client, TEMPLATE_XML, templates_dir, timeout=60)
data = lasx.parse_template_positions(
    templates_dir, TEMPLATE_BASE, client=client,
)

# ── Extract point markers (focus references) from geometries ─────────
focus_positions = [
    {"x_um": g["center_um"]["x_um"], "y_um": g["center_um"]["y_um"]}
    for g in data.get("geometries", {}).values()
    if g["type"] == "Point" and "center_um" in g
]

if not focus_positions:
    print("No point markers found.")
    print("Add them in Navigator Expert, then re-run this cell.")
else:
    print(f"Focus reference positions ({len(focus_positions)}):")
    for i, fp in enumerate(focus_positions):
        print(f"  {i + 1}. x={fp['x_um']:.1f}  y={fp['y_um']:.1f} um")

    # ── Stage limits from the positions ──────────────────────────────
    all_x = [fp["x_um"] for fp in focus_positions]
    all_y = [fp["y_um"] for fp in focus_positions]

    lasx.set_stage_limits(
        x_min=min(all_x) - LIMIT_MARGIN_UM,
        x_max=max(all_x) + LIMIT_MARGIN_UM,
        y_min=min(all_y) - LIMIT_MARGIN_UM,
        y_max=max(all_y) + LIMIT_MARGIN_UM,
        z_galvo_min=Z_GALVO_MIN,
        z_galvo_max=Z_GALVO_MAX,
        z_wide_min=Z_WIDE_MIN,
        z_wide_max=Z_WIDE_MAX,
    )

    limits = lasx.get_stage_limits()
    print(f"\nStage limits:")
    print(f"  X: {limits['x_min']:.0f} – {limits['x_max']:.0f} um")
    print(f"  Y: {limits['y_min']:.0f} – {limits['y_max']:.0f} um")

    # ── Strip template — clean slate ─────────────────────────────────
    strip_result = lasx.strip_template(client)
    assert strip_result, "Strip failed"
    print(f"\nStripped ({strip_result['original_fields']} fields removed)")

    # ── Enforce z-galvo on all jobs ──────────────────────────────────
    def _set_z_galvo_all(lrp_path):
        parsed = lasx.parse_lrp(lrp_path)
        for name in parsed["jobs"]:
            lasx.lrp_set_z_use_mode(lrp_path, "z-galvo", name)

    def _verify_z_galvo_all(lrp_path):
        parsed = lasx.parse_lrp(lrp_path)
        return all(
            lasx.lrp_verify_z_use_mode(lrp_path, "z-galvo", n)
            for n in parsed["jobs"]
        )

    result = lasx.apply_lrp_change(
        client, STRIPPED_XML, _set_z_galvo_all,
        verify_fn=_verify_z_galvo_all,
    )
    assert result and result["success"], "z-galvo enforcement failed"
    print("z-galvo enforced on all jobs")

Focus reference positions (4):
  1. x=39495.3  y=33788.7 um
  2. x=37608.2  y=53165.4 um
  3. x=86257.1  y=54039.9 um
  4. x=86993.5  y=32684.1 um

Stage limits:
  X: 37108 – 87494 um
  Y: 32184 – 54540 um

Stripped (0 fields removed)


Save timeout after 0.5s for '{ScanningTemplate}_PythonInspect_stripped.xml' (watching {ScanningTemplate}_PythonInspect_stripped.xml)
apply_lrp_change: confirm save timed out (attempt 1, timeout=0.5s)


z-galvo enforced on all jobs


## Cell 2 — Scanning field

Draw the scanning field in **Navigator Expert**, then run this cell to read and visualise it.

In [ ]:
JOB_NAME = "Overview"

# ── Save user's scan field as the main template ──────────────────────
r = lasx.save_experiment(client, TEMPLATE_XML, templates_dir, timeout=60)
assert r, "Save failed"

data = lasx.parse_template_positions(
    templates_dir, TEMPLATE_BASE, client=client,
)

# ── Use XML tiles if available, otherwise synthesize from geometries ─
positions = data.get("acquisition_positions", {})
if not positions and data.get("geometries"):
    from lasx.scanning_template_parsers import _tile_size_from_image_size_str
    settings = lasx.get_job_settings(client, JOB_NAME)
    tile_size_um = None
    if settings:
        tile_size_um = _tile_size_from_image_size_str(
            settings.get("imageSize", ""))
    if tile_size_um:
        data = lasx.synthesize_tiles(
            data, tile_size_um, job_name=JOB_NAME)
        positions = data["acquisition_positions"]
        print(f"Synthesized tiles from {len(data['geometries'])} geometries "
              f"(tile size {tile_size_um:.1f} um)")
    else:
        print("WARNING: Cannot determine tile size — no tiles generated")

n_regions = len(positions)
n_tiles = sum(len(r["positions"]) for r in positions.values())
print(f"Scanning field: {n_regions} region(s), {n_tiles} tile(s)")

for rid, region in positions.items():
    print(f"  Region {rid}: {region['job_name']}  "
          f"{region.get('num_rows', '?')}x{region.get('num_cols', '?')}  "
          f"tile={region.get('tile_size_um', '?')} um")

# ── Visualise ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor("white")
ax.set_facecolor("#f5f5f8")

all_x, all_y = [], []
viz_colors = data.get("visualization_data", {}).get("tile_colors", {})
legend_jobs = set()

# ── Build job color map from data ────────────────────────────────────
job_color_map = {}
for rid, region in positions.items():
    jn = region["job_name"]
    if jn not in job_color_map and jn in viz_colors:
        job_color_map[jn] = tuple(viz_colors[jn])

# ── Tiles ────────────────────────────────────────────────────────────
for rid, region in positions.items():
    jn = region["job_name"]
    ts = region.get("tile_size_um")
    if ts is None:
        continue
    half = ts / 2

    rgba = job_color_map.get(jn, (0.78, 0.78, 0.78, 1.0))
    face = (rgba[0], rgba[1], rgba[2], 0.25)
    edge = (rgba[0], rgba[1], rgba[2], 0.80)

    for pos in region["positions"]:
        cx, cy = pos["x_um"], pos["y_um"]
        ax.add_patch(patches.Rectangle(
            (cx - half, cy - half), ts, ts,
            linewidth=0.6, edgecolor=edge, facecolor=face, zorder=2,
        ))
        all_x.extend([cx - half, cx + half])
        all_y.extend([cy - half, cy + half])

    if jn not in legend_jobs:
        label = "No job assigned" if jn == "(unassigned)" else jn
        ax.plot([], [], "s", color=(rgba[0], rgba[1], rgba[2], 0.6),
                markersize=8, label=label)
        legend_jobs.add(jn)

# ── Stage limits ─────────────────────────────────────────────────────
try:
    lim = lasx.get_stage_limits()
    if lim:
        lw = lim["x_max"] - lim["x_min"]
        lh = lim["y_max"] - lim["y_min"]
        ax.add_patch(patches.Rectangle(
            (lim["x_min"], lim["y_min"]), lw, lh,
            linewidth=1.0, edgecolor="#aaaaaa", facecolor="none",
            linestyle=(0, (5, 4)), zorder=1,
        ))
        ax.plot([], [], ls=(0, (5, 4)), color="#aaaaaa",
                linewidth=1.0, label="Stage limits")
        all_x.extend([lim["x_min"], lim["x_max"]])
        all_y.extend([lim["y_min"], lim["y_max"]])
except Exception:
    pass

# ── Limit reference points ───────────────────────────────────────────
_fps = focus_positions if "focus_positions" in dir() else []
if _fps:
    rx = [fp["x_um"] for fp in _fps]
    ry = [fp["y_um"] for fp in _fps]
    ax.scatter(rx, ry, s=30, c="none", edgecolors="#999999",
               linewidths=0.9, zorder=8, marker="D")
    ax.plot([], [], "D", color="none", markeredgecolor="#999999",
            markersize=5, markeredgewidth=0.9, label="Limit references")
    all_x.extend(rx)
    all_y.extend(ry)

# ── Focus & autofocus points from template ───────────────────────────
cross_range = (max(max(all_x) - min(all_x), max(all_y) - min(all_y))
               if all_x else 10_000)
cross = cross_range * 0.006
circle_r = cross * 0.6

for fp_list, color, label in [
    (data.get("focus_points", []), "#4EB8B8", "Focus points"),
    (data.get("autofocus_points", []), "#5CBF5C", "AutoFocus points"),
]:
    if not fp_list:
        continue
    for fp in fp_list:
        fx, fy = fp["x_um"], fp["y_um"]
        ax.plot([fx - cross, fx + cross], [fy, fy],
                "-", color=color, linewidth=0.8, zorder=10)
        ax.plot([fx, fx], [fy - cross, fy + cross],
                "-", color=color, linewidth=0.8, zorder=10)
        ax.add_patch(patches.Circle(
            (fx, fy), circle_r,
            linewidth=0.8, edgecolor=color, facecolor="none", zorder=11))
        all_x.append(fx)
        all_y.append(fy)
    ax.plot([], [], "+", color=color, markersize=10,
            markeredgewidth=1.8, label=label)

# ── Axes ─────────────────────────────────────────────────────────────
if all_x and all_y:
    span = max(max(all_x) - min(all_x), max(all_y) - min(all_y))
    pad = span * 0.05
    ax.set_xlim(min(all_x) - pad, max(all_x) + pad)
    ax.set_ylim(min(all_y) - pad, max(all_y) + pad)

ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_xticks([])
ax.set_yticks([])
ax.grid(False)
for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_edgecolor("#cccccc")

ax.set_title("Acquisition Layout", fontsize=13, fontweight="bold",
             color="#222222", pad=12)
ax.legend(loc="upper right", fontsize=9, facecolor="white",
          edgecolor="#cccccc", labelcolor="#444444")

plt.tight_layout()
plt.show()

## Cell 3 — Autofocus

Strips the template, runs the AF job at each focus position, reads back Z, fits a plane.

In [ ]:
AF_JOB = "AF Job"
RESTORE_AFTER_AF = True

# ── Strip for autofocus ──────────────────────────────────────────────
lasx.strip_template(client)
lasx.select_job(client, AF_JOB)

# ── Measure Z at each focus position ─────────────────────────────────
measured = []
for i, fp in enumerate(focus_positions):
    print(f"[{i + 1}/{len(focus_positions)}] "
          f"x={fp['x_um']:.0f}  y={fp['y_um']:.0f}", end="", flush=True)

    lasx.move_xy(client, fp["x_um"], fp["y_um"])
    lasx.acquire(client, AF_JOB)

    settings = lasx.get_job_settings(client, AF_JOB)
    ch = lasx.make_changeable_copy(settings)
    z_um = ch["zPosition"]["z-galvo"]

    measured.append({**fp, "z_um": z_um})
    print(f"  z={z_um:.2f} um")

# ── Restore template ─────────────────────────────────────────────────
if RESTORE_AFTER_AF:
    lasx.restore_template(client)

# ── Fit Z plane: z = a*x + b*y + c ──────────────────────────────────
xs = np.array([m["x_um"] for m in measured])
ys = np.array([m["y_um"] for m in measured])
zs = np.array([m["z_um"] for m in measured])

A = np.column_stack([xs, ys, np.ones(len(measured))])
plane_coeffs, *_ = np.linalg.lstsq(A, zs, rcond=None)

def interpolate_z(x, y):
    return plane_coeffs[0] * x + plane_coeffs[1] * y + plane_coeffs[2]

residuals = zs - np.array([interpolate_z(x, y) for x, y in zip(xs, ys)])
z_range = zs.max() - zs.min()
tilt_x = np.degrees(np.arctan(plane_coeffs[0]))
tilt_y = np.degrees(np.arctan(plane_coeffs[1]))

print(f"\nFocus map:")
print(f"  Z range:      {z_range:.2f} um")
print(f"  Tilt X:       {tilt_x:+.4f} deg")
print(f"  Tilt Y:       {tilt_y:+.4f} deg")
print(f"  Max residual: {np.max(np.abs(residuals)):.3f} um")

# ── Visualise ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
sc = ax.scatter(xs, ys, c=zs, cmap="coolwarm", s=200, edgecolors="k")
for m in measured:
    ax.annotate(f"{m['z_um']:.2f}", (m["x_um"], m["y_um"]),
                textcoords="offset points", xytext=(8, 8), fontsize=8)
plt.colorbar(sc, ax=ax, label="Z (um)")
ax.set_xlabel("X (um)")
ax.set_ylabel("Y (um)")
ax.set_title("Measured Focus Points")
ax.set_aspect("equal")
ax.invert_yaxis()

ax = axes[1]
all_x = [p["x_um"] for r in positions.values() for p in r["positions"]]
all_y = [p["y_um"] for r in positions.values() for p in r["positions"]]
if all_x:
    margin = 50
    xi = np.linspace(min(all_x) - margin, max(all_x) + margin, 100)
    yi = np.linspace(min(all_y) - margin, max(all_y) + margin, 100)
    Xi, Yi = np.meshgrid(xi, yi)
    Zi = interpolate_z(Xi, Yi)
    cf = ax.contourf(Xi, Yi, Zi, levels=20, cmap="coolwarm")
    plt.colorbar(cf, ax=ax, label="Z (um)")
    ax.scatter(xs, ys, c="k", marker="*", s=100, zorder=5, label="Focus pts")
    ax.plot(all_x, all_y, ".", color="gray", markersize=2, label="Tiles")
    ax.legend(fontsize=8)
ax.set_xlabel("X (um)")
ax.set_ylabel("Y (um)")
ax.set_title("Interpolated Z Surface")
ax.set_aspect("equal")
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## Cell 4 — Acquisition

Strips the template, acquires at every tile position with interpolated Z from the focus plane.

In [ ]:
JOB_NAME = "Overview"

# ── Strip for acquisition ────────────────────────────────────────────
lasx.strip_template(client)
lasx.select_job(client, JOB_NAME)

# ── Build ordered sequence (rowwise meandering) ─────────────────────
sequence = []
for rid, region in sorted(positions.items(), key=lambda r: int(r[0])):
    rows = {}
    for p in region["positions"]:
        rows.setdefault(p["row"], []).append(p)
    for row_idx in sorted(rows):
        row_tiles = sorted(rows[row_idx], key=lambda p: p["col"])
        if row_idx % 2 == 1:
            row_tiles = row_tiles[::-1]
        for p in row_tiles:
            sequence.append({
                "region": rid,
                "x_um": p["x_um"],
                "y_um": p["y_um"],
                "z_um": interpolate_z(p["x_um"], p["y_um"]),
            })

print(f"Acquiring {len(sequence)} positions with '{JOB_NAME}'\n")

# ── Acquire ──────────────────────────────────────────────────────────
results = []
for i, pos in enumerate(sequence):
    print(f"[{i + 1}/{len(sequence)}] R{pos['region']}  "
          f"x={pos['x_um']:.0f}  y={pos['y_um']:.0f}  "
          f"z={pos['z_um']:.2f}", end="", flush=True)

    lasx.move_xy(client, pos["x_um"], pos["y_um"])
    lasx.move_z(client, JOB_NAME, pos["z_um"], z_mode="galvo")
    result = lasx.acquire(client, JOB_NAME)

    results.append({**pos, "success": result["success"]})
    elapsed = result["timing"]["total_s"] if result["success"] else 0
    status = "OK" if result["success"] else "FAIL"
    print(f"  {status} ({elapsed:.1f}s)")

ok = sum(1 for r in results if r["success"])
print(f"\nDone: {ok}/{len(results)} successful")